# 🏆 YOLOv26 Night Detection Fine-Tuning (NVIDIA B200 GPU Edition)
This notebook will guide you through the process of fine-tuning YOLOv26-Nano to improve vehicle and person detection on night CCTV feeds.

### 1. Verify PyTorch and B200 GPU Status
Let's make sure PyTorch is installed and can access the NVIDIA B200 GPU.

In [ ]:
import torch
import sys
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count:", torch.cuda.device_count())
    print("Device Name:", torch.cuda.get_device_name(0))

### 2. Prepare the dataset.yaml configuration
Create the dataset configuration specifying the classes you want to train. This script programmatically creates `dataset/dataset.yaml`.

In [ ]:
import yaml
import os

# Define the dataset configuration
dataset_config = {
    "path": "/root/workspace/geo/dataset",
    "train": "train/images",
    "val": "val/images",
    "names": {
        0: "person",
        2: "car",
        3: "motorcycle",
        5: "bus",
        7: "truck"
    }
}

# Ensure dataset directory exists
os.makedirs("dataset", exist_ok=True)
with open("dataset/dataset.yaml", "w") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print("✅ Created dataset/dataset.yaml successfully!")

### 3. Acquire Training Data (Choose One Option)
Choose one of the following methods to populate the `dataset/` directory.

#### 📌 Option A: Auto-Collect & Auto-Label from the Live CCTV Stream (Recommended)
This script connects to your live camera feed, captures 50 frames, and uses the pre-trained YOLOv26 model to automatically generate annotations (labels) at `conf=0.15`. This creates a custom dataset tailored specifically for your camera view.

In [ ]:
import cv2
import os
import time
from ultralytics import YOLO

camera_url = "https://camera1.iticfoundation.org/mjpeg2.php?camid=10.8.0.14:8001"

# Create directories
os.makedirs("dataset/train/images", exist_ok=True)
os.makedirs("dataset/train/labels", exist_ok=True)
os.makedirs("dataset/val/images", exist_ok=True)
os.makedirs("dataset/val/labels", exist_ok=True)

# Load our current model on GPU
model = YOLO("yolo26n.pt")
model.to('cuda:0')

print("🟢 Starting auto-collection of 50 frames from CCTV...")
for i in range(50):
    cap = cv2.VideoCapture(camera_url)
    ret, frame = cap.read()
    cap.release()
    
    if not ret or frame is None:
        print(f"[{i+1}/50] Failed to fetch frame, retrying...")
        time.sleep(1.0)
        continue
        
    # Split 80% train / 20% val
    split = "train" if i < 40 else "val"
    img_name = f"frame_{i:03d}.jpg"
    img_path = f"dataset/{split}/images/{img_name}"
    label_path = f"dataset/{split}/labels/frame_{i:03d}.txt"
    
    # Save image frame
    cv2.imwrite(img_path, frame)
    
    # Run YOLO to auto-label vehicles/people
    results = model(frame, conf=0.15, verbose=False)
    h, w, _ = frame.shape
    
    with open(label_path, "w") as f:
        for r in results:
            for box in r.boxes:
                cls_id = int(box.cls[0])
                if cls_id in [0, 2, 3, 5, 7]: # person, car, motorcycle, bus, truck
                    xyxy = box.xyxy[0].tolist()
                    # Convert to relative center x, center y, width, height (YOLO format)
                    cx = ((xyxy[0] + xyxy[2]) / 2.0) / w
                    cy = ((xyxy[1] + xyxy[3]) / 2.0) / h
                    bw = (xyxy[2] - xyxy[0]) / w
                    bh = (xyxy[3] - xyxy[1]) / h
                    f.write(f"{cls_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
                    
    print(f"[{i+1}/50] Saved and auto-labeled: {img_name} ({split})")
    time.sleep(1.0) # Wait 1 second to capture different traffic frames

print("\n✅ Done! Auto-labeled dataset created at 'dataset/' folder.")

#### 📌 Option B: Download COCO128 Dataset
If you want to train on a standard reference dataset immediately, this downloads the COCO128 dataset (128 images of cars, motorcycles, people, etc. with pre-labeled bounding boxes).

In [ ]:
import urllib.request
import zipfile
import shutil
import os

url = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip"
zip_path = "coco128.zip"

print("🟢 Downloading COCO128 dataset...")
urllib.request.urlretrieve(url, zip_path)

print("🟢 Extracting dataset...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")

# Clean up existing dataset directory and replace it
if os.path.exists("dataset"):
    shutil.rmtree("dataset")
os.rename("coco128", "dataset")
os.remove(zip_path)

print("\n✅ COCO128 dataset successfully downloaded and extracted to 'dataset/'!")

### 4. Start Training YOLOv26-Nano
We will load the pretrained YOLOv26-Nano model (`yolo26n.pt`) and train it on the B200 GPU (`device=0`).

In [ ]:
from ultralytics import YOLO

# Load pretrained model weights
model = YOLO("yolo26n.pt")

# Start training
results = model.train(
    data="dataset/dataset.yaml",
    epochs=50,                  # Adjust number of epochs as needed
    imgsz=640,                  # Image size
    device=0,                   # NVIDIA B200 GPU
    batch=16,                   # Batch size
    workers=4,
    project="yolov8_night_cctv",
    name="train_run"
)

### 5. Evaluate the model
Run validation to check precision, recall, and mAP metrics on your validation set.

In [ ]:
# Validate the model
metrics = model.val()
print("mAP 50-95:", metrics.box.map)
print("mAP 50:", metrics.box.map50)

### 6. Deploy the trained model
Copy the best-performing weights (`best.pt`) to the backend folder (`backend/yolo26n.pt`) to replace the default model.

In [ ]:
import shutil
import os

best_weights = "yolov8_night_cctv/train_run/weights/best.pt"
target_model = "backend/yolo26n.pt"

if os.path.exists(best_weights):
    shutil.copy(best_weights, target_model)
    print(f"✅ Successfully copied and deployed new weights to: {target_model}!")
    print("To load the new weights, please restart your Uvicorn backend server.")
else: 
    print(f"❌ Could not find trained weights at {best_weights}. Please verify training logs.")